In [1]:
!pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 73.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 120.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 85.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 114.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/19

In [1]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Current GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Create a specific folder for your research project
save_path = "/content/drive/MyDrive/Research_SLM_Baseline"
if not os.path.exists(save_path):
    os.makedirs(save_path)

Mounted at /content/drive


In [3]:
!pip install transformers peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Setup paths (Change folder_name if you saved it differently)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "/content/drive/MyDrive/Research_SLM_Baseline/baseline_round_3"

print("Loading base model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 2. Load the trained adapters
print("Loading trained Federated adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# 3. Define the refined chat/inference function
def generate_response(prompt, max_new_tokens=150):
    # Wrap the prompt in TinyLlama's chat format
    # This signals the transition from User to Assistant
    formatted_prompt = f"<|system|>\nYou are a helpful assistant.</s>\n<|user|>\n{prompt}</s>\n<|assistant|>\n"

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.2, # Direct fix for echoing/looping
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract only the NEW tokens (the answer)
    # This slices the output array to start where the input array ended
    prompt_length = inputs.input_ids.shape[1]
    new_tokens = outputs[0][prompt_length:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# 4. TEST IT OUT
test_prompt = "Explain the concept of Federated Learning in one sentence."
print(f"\n--- Prompt ---\n{test_prompt}")

response = generate_response(test_prompt)
print(f"\n--- Model Response ---\n{response}")

Loading base model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading trained Federated adapters...

--- Prompt ---
Explain the concept of Federated Learning in one sentence.

--- Model Response ---
Federated learning is a distributed data processing technique that enables decentralized computation, allowing multiple parties to collaboratively train machine learning models without sharing sensitive or confidential data. It uses public cloud services for storage and computing resources rather than having all data stored on central servers. The process works by distributing training datasets across different users with their private key/salt information, which allows each user's training task to be processed independently but still aggregate together as part of a larger federation. This methodology ensures security while reducing costs and improving scalability compared to traditional centralized systems.
